# Lab 9 — Eigenvalues via Rayleigh Quotients
**Linear Algebra · UATX**  
*Week 9 · Prerequisites: §4.2 (pages 35–38)*

For a symmetric matrix $A$, the **Rayleigh quotient** $R(x) = x^\top A x / x^\top x$ has minimum $\lambda_{\min}$ and maximum $\lambda_{\max}$ over the unit sphere. This motivates finding eigenvalues by *gradient descent on the sphere*, then **deflating** (projecting out found eigenvectors) to recover the full spectrum — without ever computing $\det(A - \lambda I)$.

**Tasks**
1. Implement gradient descent on the sphere for the smallest eigenvalue.
2. Recover the full spectrum by deflation; measure $L^2$ error vs `numpy.linalg.eigh`.
3. *(New)* Plot convergence $|R(x_t) - \lambda_{\min}|$ for different learning rates.
4. *(New)* Time the algorithm on a $100 \times 100$ matrix vs `numpy.linalg.eigh`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
np.random.seed(0)

# Build a 10x10 random symmetric matrix
n = 10
M = np.random.randn(n, n)
A = M + M.T

true_eigs, _ = np.linalg.eigh(A)
print('True eigenvalues (sorted):', np.round(true_eigs, 4))

## §1  Gradient Descent on the Unit Sphere

The gradient of $R(x)$ restricted to the unit sphere (the Riemannian gradient) is $2(Ax - R(x)\,x)$. We minimise $R$ by gradient descent followed by re-normalisation.

In [ ]:
def rayleigh_quotient(A, x):
    return (x @ (A @ x)) / (x @ x)

def min_eigen_via_gradient(A, x0, lr=0.01, steps=500):
    x = x0 / np.linalg.norm(x0)
    for _ in range(steps):
        R = rayleigh_quotient(A, x)
        grad = 2 * (A @ x - R * x)
        x = x - lr * grad
        x = x / np.linalg.norm(x)
    lam = rayleigh_quotient(A, x)
    return lam, x

x0 = np.random.randn(n)
lam1, v1 = min_eigen_via_gradient(A, x0, lr=0.01, steps=500)
print(f'Approx. smallest eigenvalue: {lam1:.8f}')
print(f'True  smallest eigenvalue:   {true_eigs[0]:.8f}')
print(f'Match (atol=1e-3): {np.isclose(lam1, true_eigs[0], atol=1e-3)}')

## §2  Full Spectrum via Deflation

Once eigenvector $v_1$ is found, project it out: minimise $R$ over $\{x : x \perp v_1\}$ to find $v_2$, then over $\{x \perp v_1, v_2\}$ for $v_3$, etc.

In [ ]:
def next_eigen_via_deflation(A, prev_vecs, lr=0.01, steps=500):
    n = A.shape[0]
    V = np.column_stack(prev_vecs) if prev_vecs else np.zeros((n, 0))
    x = np.random.randn(n)
    if V.shape[1] > 0: x -= V @ (V.T @ x)
    x /= np.linalg.norm(x)
    for _ in range(steps):
        R = rayleigh_quotient(A, x)
        grad = 2 * (A @ x - R * x)
        if V.shape[1] > 0: grad -= V @ (V.T @ grad)
        x -= lr * grad
        if V.shape[1] > 0: x -= V @ (V.T @ x)
        x /= np.linalg.norm(x)
    return rayleigh_quotient(A, x), x

def compute_k_smallest(A, k, lr=0.01, steps=1200):
    eigs, vecs = [], []
    for _ in range(k):
        lam, v = next_eigen_via_deflation(A, vecs, lr=lr, steps=steps)
        eigs.append(lam); vecs.append(v)
    return np.array(eigs), np.column_stack(vecs)


lam_gd, V_gd = compute_k_smallest(A, k=n)
print('NumPy  eigs:', np.round(true_eigs, 6))
print('GD     eigs:', np.round(lam_gd, 6))
print(f'L2 error: {np.mean((lam_gd - true_eigs)**2):.2e}')

## §3  Convergence Analysis  *(New)*

How fast does $|R(x_t) - \lambda_{\min}|$ decrease? Does the learning rate $\eta$ affect speed or stability?

In [ ]:
def track_convergence(A, lr, steps=600, seed=42):
    np.random.seed(seed)
    x = np.random.randn(A.shape[0])
    x /= np.linalg.norm(x)
    lam_min = np.linalg.eigh(A)[0][0]
    errors = []
    for _ in range(steps):
        R = rayleigh_quotient(A, x)
        errors.append(abs(R - lam_min))
        grad = 2 * (A @ x - R * x)
        x = x - lr * grad
        x /= np.linalg.norm(x)
    return errors

learning_rates = [0.001, 0.005, 0.01, 0.05, 0.1]
plt.figure(figsize=(8, 4))
for lr in learning_rates:
    errors = track_convergence(A, lr)
    # Clip at 1e-12 for log scale
    errors_clipped = [max(e, 1e-12) for e in errors]
    plt.semilogy(errors_clipped, label=f'$\\eta={lr}$')
plt.xlabel('Step'); plt.ylabel('$|R(x_t) - \\lambda_{\\min}|$')
plt.title('Convergence of Rayleigh quotient gradient descent')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## §4  Scaling to $100 \times 100$  *(New)*

For large matrices, `numpy.linalg.eigh` uses LAPACK routines (implicitly shifted QR, $O(n^3)$). Our gradient-descent method can be competitive if we only need a few eigenvalues, but it also costs $O(n^2)$ per step.

In [ ]:
n_big = 100
np.random.seed(5)
M_big = np.random.randn(n_big, n_big)
A_big = M_big + M_big.T

# NumPy timing
t0 = time.perf_counter()
eigs_np, _ = np.linalg.eigh(A_big)
t_numpy = time.perf_counter() - t0

# GD for k=3 smallest eigenvalues
t0 = time.perf_counter()
k = 3
lam_gd_big, _ = compute_k_smallest(A_big, k=k, lr=0.003, steps=1500)
t_gd = time.perf_counter() - t0

print(f'n={n_big}')
print(f'NumPy eigh (all {n_big} eigs): {t_numpy*1000:.1f} ms')
print(f'GD deflation (top {k} eigs):  {t_gd*1000:.1f} ms')
print(f'\nTrue {k} smallest: {np.round(eigs_np[:k], 4)}')
print(f'GD   {k} smallest: {np.round(lam_gd_big, 4)}')
print(f'L2 error: {np.mean((lam_gd_big - eigs_np[:k])**2):.2e}')

In [ ]:
# When is GD preferred?
# Answer: when we only need a FEW extreme eigenvalues and the matrix is very large
# (LAPACK eigh computes ALL eigenvalues; GD+deflation avoids this)
# For small k, GD costs O(k * steps * n^2) vs O(n^3) for eigh.

# Verify eigenvectors are orthonormal
lam_gd_all, V_gd_all = compute_k_smallest(A_big, k=n_big, lr=0.003, steps=1000)
print('Eigenvector matrix V^T V ~ I?', np.allclose(V_gd_all.T @ V_gd_all, np.eye(n_big), atol=1e-2))
print('Max off-diagonal of V^T V:', np.max(np.abs(V_gd_all.T @ V_gd_all - np.eye(n_big))))